In [1]:
import os
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
import torch

In [2]:
environment_name = "CartPole-v1"

**Probamos el environment sin modelo**

In [3]:
env = gym.make(environment_name,render_mode="human")

In [ ]:
episodes = 5
for episode in range(1, episodes+1):
    state, info = env.reset()
    done = False
    score = 0 
    
    while not done:
        action = env.action_space.sample()
        n_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        
        score += reward
        
    print(f'Episode:{episode} Score:{score}')


In [ ]:
env.reset()

In [ ]:
env.step(1)

In [ ]:
env.action_space

In [ ]:
env.action_space.sample()

In [ ]:
env.observation_space

In [ ]:
env.observation_space.sample()

In [11]:
env.close()

**Aplicamos un modelo**

In [12]:
# Definimos una función para crear el entorno
def make_env():
    return gym.make(environment_name)

In [13]:
# Creamos el entorno vectorizado correctamente
env = DummyVecEnv([make_env])

In [ ]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

In [ ]:
# Definimos el modelo
# MlpPolicy: Red neuronal estándar (Multi-layer Perceptron)
model = PPO('MlpPolicy', env, device=device, verbose=1)

In [16]:
#model.learn(total_timesteps=20000)

In [17]:
PPO_path = os.path.join('Training', 'models', 'PPO_cart')

In [18]:
model.save(PPO_path)

In [19]:
del model

In [20]:
model = PPO.load(PPO_path, env=env)

In [21]:
from stable_baselines3.common.evaluation import evaluate_policy

In [22]:
env_eval = gym.make(environment_name,render_mode="human")

In [ ]:
evaluate_policy(model, env_eval, n_eval_episodes=10)

**Probamos el modelo**

In [ ]:
# 1. Reset devuelve (obs, info)
obs, info = env.reset() 

while True:
    # 2. Predict devuelve la acción que el modelo cree mejor
    action, _states = model.predict(obs, deterministic=True)
    
    # 3. step() devuelve 5 valores en Gymnasium
    obs, reward, terminated, truncated, info = env.step(action)
    
    # env.render() no es necesario aquí si usaste render_mode="human" al crear env
    
    # 4. Combinamos los flags de finalización
    if terminated or truncated:
            obs, info = env.reset() # Reinicia para la siguiente partida


**Entranamos ahora pero utilizando Callbacks**

In [50]:
from stable_baselines3.common.callbacks import EvalCallback, StopTrainingOnRewardThreshold

In [51]:
save_path = os.path.join('Training', 'models')

In [52]:
# Definimos una función para crear el entorno
def make_env():
    return gym.make(environment_name)
env = DummyVecEnv([make_env])

In [53]:
stop_callback = StopTrainingOnRewardThreshold(reward_threshold=400, verbose=1)
eval_callback = EvalCallback(env, 
                             callback_on_new_best=stop_callback, 
                             eval_freq=10000, 
                             best_model_save_path=save_path, 
                             verbose=1)

In [ ]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Usando dispositivo: {device}")
model = PPO('MlpPolicy', env, device=device, verbose=1)

In [ ]:
#model.learn(total_timesteps=50000, callback=eval_callback)

In [55]:
model_path = os.path.join('Training', 'models', 'best_model')
model = PPO.load(model_path, env=env)

In [ ]:
# Evaluamos el modelo
env_eval = gym.make(environment_name,render_mode="human")
evaluate_policy(model, env_eval, n_eval_episodes=10)